# 🚀 CRAFT Phase 2: Curriculum‑guided Reinforced Adaptive Fine‑Tuning (RL Loop)

This notebook trains a small language model (Phi‑3‑Mini) using **GRPO** with three integrated components:

- **Component A** – Deterministic execution verifier (math / logic scoring)
- **Component B** – Step‑level contrastive DPO (activates at step 100)
- **Component C** – Adaptive curriculum (difficulty zone 40‑70%, capped at 85%)

All critical stability fixes (gradient clipping, memory cleanup, KL warmup, safe advantage normalization) are already applied.

## 1. Setup & Dependencies

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

# Clone your private repository using GitHub PAT
user_secrets = UserSecretsClient()
git_token = user_secrets.get_secret("GITHUB_PAT")
username = "VishalGastu30"

!git clone https://{git_token}@github.com/{username}/CRAFT-SLM-Reasoning.git
%cd CRAFT-SLM-Reasoning

# Install required packages
!pip install -q transformers bitsandbytes accelerate datasets peft trl loguru pyyaml scipy numpy sentence-transformers

print("✅ Setup complete")

## 2. Load Phase 0 & Phase 1 Artifacts

Copies `difficulty_map.json` (Phase 0) and the SFT checkpoint (Phase 1) from attached Kaggle datasets.

In [ ]:
import os
import shutil

# ------------------------------------------------------------------
# 1. Find and copy Phase 0 difficulty_map.json
# ------------------------------------------------------------------
diff_map = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'difficulty_map.json' in files:
        diff_map = os.path.join(root, 'difficulty_map.json')
        break

if diff_map:
    os.makedirs("data", exist_ok=True)
    shutil.copy(diff_map, "data/difficulty_map.json")
    shutil.copy(diff_map, "difficulty_map.json")
    print("✅ Copied difficulty_map.json")
else:
    print("❌ difficulty_map.json not found! Attach Phase 0 dataset.")

# ------------------------------------------------------------------
# 2. Find and copy Phase 1 SFT checkpoint (final)
# ------------------------------------------------------------------
sft_final_dir = None

# First, search for the 'final' directory that contains adapter_config.json
for root, dirs, files in os.walk('/kaggle/input'):
    if 'final' in dirs and 'sft' in root:
        potential_path = os.path.join(root, 'final')
        if os.path.exists(os.path.join(potential_path, 'adapter_config.json')):
            sft_final_dir = potential_path
            break

if sft_final_dir:
    os.makedirs("checkpoints/sft", exist_ok=True)
    if not os.path.exists("checkpoints/sft/final"):
        shutil.copytree(sft_final_dir, "checkpoints/sft/final")
        print(f"✅ Copied SFT checkpoint from {sft_final_dir}")
    else:
        print("✅ SFT checkpoint already exists")
else:
    print("❌ SFT checkpoint not found! Searching manually for adapter_config.json...")
    # Fallback: search for adapter_config.json anywhere
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'adapter_config.json' in files:
            sft_final_dir = root
            print(f"Found adapter_config.json at: {root}")
            break

    if sft_final_dir:
        os.makedirs("checkpoints/sft", exist_ok=True)
        shutil.copytree(sft_final_dir, "checkpoints/sft/final")
        print("✅ Copied SFT checkpoint (fallback)")
    else:
        print("❌ Still not found. Make sure Phase 1 dataset is attached.")

# ------------------------------------------------------------------
# 3. Verify both required artifacts exist
# ------------------------------------------------------------------
assert os.path.exists("checkpoints/sft/final"), "Missing SFT checkpoint"
assert os.path.exists("difficulty_map.json"), "Missing difficulty map"
print("✅ All required artifacts are ready.")

## 3. (Optional) Load Previous RL Checkpoints

If you want to resume training from a previously saved checkpoint, attach the RL output dataset. The script will automatically detect and copy `checkpoint-XXX` folders.

In [ ]:
"""
import os
import shutil

rl_checkpoints = []

for root, dirs, files in os.walk('/kaggle/input'):
    # Normalize path separators to forward slashes for consistent matching
    norm_root = root.replace('\\', '/')
    # Only consider checkpoints that live under a directory named 'rl'
    if '/rl/' in norm_root or norm_root.endswith('/rl'):
        for d in dirs:
            if d.startswith('checkpoint-'):
                src_path = os.path.join(root, d)
                rl_checkpoints.append((src_path, d))

if rl_checkpoints:
    dest = "checkpoints/rl"
    os.makedirs(dest, exist_ok=True)
    for src_path, folder_name in rl_checkpoints:
        dst = os.path.join(dest, folder_name)
        if not os.path.exists(dst):
            print(f"Copying {folder_name} from {src_path} ...")
            shutil.copytree(src_path, dst)
        else:
            print(f"⏩ {folder_name} already exists, skipping.")
    print("\n✅ RL checkpoints ready. Use --resume to continue training.")
else:
    print("ℹ️ No previous RL checkpoints found. Starting fresh.")
"""

## 4. Run Pre‑Training Diagnostics

Verifies reward scorer, KL controller warmup, and GPU availability.

In [ ]:
!python -m src.phase2_rl.run_diagnostics

## 5. Start RL Training

- **First run (fresh)**: Remove `--resume` flag.
- **Resume from last checkpoint**: Keep `--resume` flag.

Training will save checkpoints every 50 steps. Expected duration: **3‑4 hours**.

In [ ]:
# Set RESUME = True if you want to continue from the latest checkpoint
RESUME = False   # Change to True when you have a checkpoint

if RESUME:
    !python -m src.phase2_rl.craft_rl_loop --config phi3_mini --hardware kaggle --output checkpoints/rl --resume
else:
    !python -m src.phase2_rl.craft_rl_loop --config phi3_mini --hardware kaggle --output checkpoints/rl

## 6. Verify Output

After training completes (500 steps), the final model is saved to `checkpoints/rl/final`.

In [ ]:
final_path = "checkpoints/rl/final"
if os.path.exists(final_path):
    print(f"✅ Training completed. Final model saved at: {final_path}")
    print("Contents:", os.listdir(final_path))
else:
    print("❌ Final model not found. Training may have been interrupted.")
    print("Check the logs above for errors.")